In [0]:
# -------------------------------
# Spark Session Initialization and create or get 
# -------------------------------
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.session.timeZone", "UTC")

import os
env = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_ENV")
pipeline_name = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_NAME")
pipeline_version = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_VERSION")

os.environ["SUPERSTORE_ENV"] = env
os.environ["SUPERSTORE_PIPELINE_NAME"] = pipeline_name
os.environ["SUPERSTORE_PIPELINE_VERSION"] = pipeline_version

import sys
# -----------------------------
# Add shared utilities path
# -----------------------------
# importing config file 
sys.path.append("/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/src/superstore_shared_utilities")
# importing module
sys.path.append("/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/src/superstore_bronze")

# -----------------------------
# Import logging and orchestration utilities
# -----------------------------
from superstore_logger import get_superstore_logger, log_event
from superstore_orchestrator_execution_context import get_execution_context
from superstore_platform_constants import BRONZE_LAYER
from superstore_platform_config import get_catalog, get_bronze_schema, get_metrics_schema, table  # << Platform imports

# -----------------------------
# Receive master_run_id & generate layer_run_id
# -----------------------------
execution_context = get_execution_context(dbutils, allow_standalone=True)
master_run_id = execution_context["master_run_id"]
layer_run_id = execution_context["layer_run_id"]

# -----------------------------
# Initialize Logger for this child layer
# -----------------------------
logger_bronze_orchestrator = get_superstore_logger("superstore_bronze_layer_ETL_orchestrator")

import yaml

config_file_path = "/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/configs/superstore_bronze_config/superstore_bronze_config.yaml"

with open(config_file_path, "r") as f:
    config = yaml.safe_load(f)

# -----------------------------
# # Access paths dynamically for current environment
# -----------------------------
raw_source_file_path = config['env_paths'][env]['raw_source_file_path']
schema_location = config['env_paths'][env]['schema_location']
checkpoint_location = config['env_paths'][env]['checkpoint_location']

# -----------------------------
# Log layer start
# -----------------------------
log_event(
    logger_bronze_orchestrator,
    "INFO",
    "Initiating Superstore Bronze layer ETL orchestrator",
    master_run_id=master_run_id,
    layer_run_id=layer_run_id,
    layer=BRONZE_LAYER
)

from pyspark.sql.functions import col
import importlib
import bronze_ingest_superstore_module_01
importlib.reload(bronze_ingest_superstore_module_01)
from bronze_ingest_superstore_module_01 import bronze_ingest_incremental

import bronze_entity_superstore_module_02
importlib.reload(bronze_entity_superstore_module_02)
from bronze_entity_superstore_module_02 import get_last_processed_ts, bronze_entity_incremental_append, optimize_zorder_bronze_tables, collect_entity_metrics

from datetime import datetime
import pytz
import time
from pyspark.sql import DataFrame
import yaml

# -----------------------------
# Load YAML Bronze Config
# -----------------------------
config_path = "/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/configs/superstore_bronze_config/superstore_bronze_config.yaml"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

COLUMN_RENAME_MAP = config["column_rename_map"]
BRONZE_MAIN = config["bronze_main"]
BRONZE_ENTITIES = config["bronze_entities"]

# -----------------------------
# Bronze Layer Runner
# -----------------------------
def run_bronze_layer(spark, master_run_id: str, raw_source_file_path:str, schema_location:str, checkpoint_location:str):

    # Use platform config to resolve catalog and schema
    bronze_raw_table = table(get_bronze_schema(), BRONZE_MAIN["table_name"])

    pipeline_start = time.time()

    # -----------------------------
    # 1. Bronze Ingest (Incremental)
    # -----------------------------
    try:
       bronze_ingest_incremental(
            spark=spark,
            raw_source_file_path=raw_source_file_path,
            schema_location=schema_location,
            checkpoint_location=checkpoint_location,
            column_rename_map=COLUMN_RENAME_MAP,
            metadata_columns=BRONZE_MAIN["metadata_columns"],
            table_name=bronze_raw_table,  
            master_run_id=master_run_id,
            layer_run_id=layer_run_id
        )
    except Exception as e:
        log_event(
            logger_bronze_orchestrator,
            "ERROR",
            f"Bronze ingest failed: {e}",
            exc_info=True,
            master_run_id=master_run_id,
            layer_run_id=layer_run_id,
            layer=BRONZE_LAYER
        )
        raise

    # -----------------------------
    # 2. Loop over Bronze Entities
    # -----------------------------
    for entity_name, cfg in BRONZE_ENTITIES.items():

        metadata_columns = cfg.get("META_COLUMNS", [])
        metrics_table = table(get_metrics_schema(), cfg["metrics_table"].split('.')[-1])  # dynamically resolve catalog+schema
        entity_table_name = table(get_bronze_schema(), cfg["table_name"].split('.')[-1])
        source_table = table(get_bronze_schema(), cfg["source_table"].split('.')[-1])
        layer_name = cfg["layer_name"]
        z_order_cols= cfg["z_order_columns"]

        start_ts = spark.sql("SELECT current_timestamp() as ts").first()["ts"]
        start_time_epoch = time.time()
        run_status = "SUCCESS"
        notes = ""

        try:
            bronze_df = spark.table(bronze_raw_table).select('*')
            last_processed_ts = get_last_processed_ts(spark, entity_table_name)

            if last_processed_ts:
                incremental_df = bronze_df.filter(col("ingestion_ts") > last_processed_ts)
            else:
                incremental_df = bronze_df

            has_incremental_rows = incremental_df.limit(1).count()

            if not has_incremental_rows:
                log_event(
                    logger_bronze_orchestrator,
                    "INFO",
                    f"Entity '{entity_name}' has no new rows/files for this layer run; skipping Bronze append.",
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=BRONZE_LAYER
                )
                empty_df = spark.createDataFrame([], bronze_df.schema)
                collect_entity_metrics(
                    spark=spark,
                    entity_df=empty_df,
                    metrics_table=metrics_table,
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer_name=layer_name,
                    source_table=source_table,
                    entity_name=entity_name,
                    target_table=entity_table_name,
                    start_ts=start_ts,
                    end_ts=start_ts,
                    duration_secs=0,
                    run_status="skipped",
                    notes="No new rows/files ingested"
                )
            else:
                df_entity = bronze_entity_incremental_append(
                    entity_name=entity_name,
                    entity_cfg=cfg,
                    spark=spark,
                    metadata_columns=metadata_columns,
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id
                )

                end_ts = spark.sql("SELECT current_timestamp() as ts").first()["ts"]
                duration_secs = int(time.time() - start_time_epoch)

                collect_entity_metrics(
                    spark=spark,
                    entity_df=df_entity,
                    metrics_table=metrics_table,
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer_name=layer_name,
                    source_table=source_table,
                    entity_name=entity_name,
                    target_table=entity_table_name,
                    start_ts=start_ts,
                    end_ts=end_ts,
                    duration_secs=duration_secs,
                    run_status=run_status,
                    notes=notes
                )

                optimize_zorder_bronze_tables(
                    spark=spark,
                    # bronze_entities={entity_table_name: cfg},
                    entity_name=entity_name,
                    entity_table_name=entity_table_name,
                    z_order_cols=z_order_cols,
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id
                )

                log_event(
                    logger_bronze_orchestrator,
                    "INFO",
                    f"Completed Bronze entity processing: {entity_name}",
                    master_run_id=master_run_id,
                    layer_run_id=layer_run_id,
                    layer=BRONZE_LAYER
                )

        except Exception as e:
            run_status = "FAILED"
            notes = str(e)
            log_event(
                logger_bronze_orchestrator,
                "ERROR",
                f"Bronze entity pipeline failed for {entity_name}: {e}",
                exc_info=True,
                master_run_id=master_run_id,
                layer_run_id=layer_run_id,
                layer=BRONZE_LAYER
            )
            raise

    # -----------------------------
    # Pipeline End
    # -----------------------------
    pipeline_end = time.time()
    log_event(logger_bronze_orchestrator, 
              "INFO", f"Bronze unified pipeline completed successfully. Total duration: {round(pipeline_end - pipeline_start, 2)}s", 
              master_run_id=master_run_id,
              layer_run_id=layer_run_id,
              layer=BRONZE_LAYER
            )

# -----------------------------
# Call Bronze Layer Inline
# -----------------------------
if __name__ == "__main__":
    run_bronze_layer(spark, master_run_id, raw_source_file_path, schema_location, checkpoint_location)
